### Alvos de classificação do IDEB (Painel e nota bruta)

A limpeza (`02_data_cleaning`) remove as colunas `Painel: ...` e a nota bruta das features. Este notebook guarda esses alvos em um arquivo separado, ligado ao município pelo `id_municipio`, para os notebooks de predição usarem. Rode uma vez, sem precisar refazer a limpeza.

In [1]:
from pathlib import Path
import pandas as pd

base_path = Path.cwd().parent
raw_path = base_path / "data" / "raw" / "Base_de_Dados_IDSC-BR_2025.xlsx"
out_path = base_path / "data" / "processed" / "Base_de_Alvos_IDEB.csv"

df = pd.read_excel(raw_path, sheet_name="Todos os Dados")

colunas = {
    'ID': 'id_municipio',
    'SDG4_15_IDEB_AF': 'ideb_af_bruto',
    'SDG4_16_IDEB_AI': 'ideb_ai_bruto',
    'Painel: SDG4_15_IDEB_AF': 'painel_af',
    'Painel: SDG4_16_IDEB_AI': 'painel_ai',
}

alvos = df[list(colunas)].rename(columns=colunas).set_index('id_municipio')
alvos.to_csv(out_path, index_label='id_municipio')

print(f"Salvo em: {out_path}  ({len(alvos)} municípios)")
alvos.head()


Salvo em: /home/clara/Desktop/idsc-analysis/data/processed/Base_de_Alvos_IDEB.csv  (5570 municípios)


,ideb_af_bruto,ideb_ai_bruto,painel_af,painel_ai
id_municipio,,,,
1100015,4.0,5.1,orange,orange
1100023,4.4,5.3,orange,orange
1100031,NaN,5.8,NaN,yellow
1100049,4.0,5.4,orange,orange
1100056,NaN,5.6,NaN,orange


Conferência: distribuição das classes e baseline (acurácia de quem sempre chuta a classe majoritária)

In [2]:
import numpy as np

for ab in ['af', 'ai']:
    print(f"--- Painel {ab.upper()} ---")
    dist = alvos[f'painel_{ab}'].value_counts(normalize=True)
    print(dist.round(3).to_string())
    print(f"Baseline: {dist.max():.3f}  | sem Painel: {alvos[f'painel_{ab}'].isna().sum()} municípios\n")

    nota_x10 = (alvos[f'ideb_{ab}_bruto'] * 10).round(4)
    faixas = pd.cut(nota_x10, bins=[-np.inf, 40, 50, 60, 70, np.inf], right=False,
                    labels=['<4,0', '4,0-4,9', '5,0-5,9', '6,0-6,9', '>=7,0'])
    dist = faixas.value_counts(normalize=True).sort_index()
    print(f"--- 5 faixas na nota bruta {ab.upper()} ---")
    print(dist.round(3).to_string())
    print(f"Baseline: {dist.max():.3f}\n")


--- Painel AF ---
painel_af
yellow    0.348
orange    0.337
green     0.205
red       0.110
Baseline: 0.348  | sem Painel: 2038 municípios

--- 5 faixas na nota bruta AF ---
ideb_af_bruto
<4,0       0.240
4,0-4,9    0.445
5,0-5,9    0.279
6,0-6,9    0.031
>=7,0      0.005
Baseline: 0.445

--- Painel AI ---
painel_ai
yellow    0.400
orange    0.297
green     0.169
red       0.133
Baseline: 0.400  | sem Painel: 215 municípios

--- 5 faixas na nota bruta AI ---
ideb_ai_bruto
<4,0       0.022
4,0-4,9    0.191
5,0-5,9    0.331
6,0-6,9    0.365
>=7,0      0.092
Baseline: 0.365

